In [1]:
!uv pip install transformers accelerate torch sentencepiece huggingface_hub pyyaml --upgrade

Using Python 3.14.6 environment at: c:\Users\user\Downloads\AI-Training\.venv
Resolved 38 packages in 996ms
Checked 38 packages in 13ms


In [ ]:
# Task 4: Chain of Thought Prompting

import os
import warnings
import logging
import torch
import re

from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Suppress Warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Load Environment Variables
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Model Name
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=HF_TOKEN,
    torch_dtype=torch.float32
)

# Create Hugging Face Pipeline
chatbot = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

# CHAIN OF THOUGHT PROMPT DESIGN
system_prompt = """
You are a medical AI assistant.

A patient has the following symptoms:
- Fever
- Cough
- Oxygen Level = 88%

Evaluate the patient's condition step-by-step. 
1. First, analyze the severity of each symptom (especially the oxygen level).
2. Second, consider potential medical emergencies.
3. Third, formulate a recommendation.

IMPORTANT: You MUST write your step-by-step reasoning inside <reasoning> and </reasoning> tags. 
After the closing </reasoning> tag, you must provide your final output starting with "Final Recommendation:".
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Please evaluate the patient and give your recommendation."}
]

# Generate Response
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

response = chatbot(
    prompt,
    max_new_tokens=300,
    do_sample=False,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id
)

# Extract raw answer
raw_answer = response[0]["generated_text"][len(prompt):].strip()

# ==========================================================
# HIDE INTERNAL REASONING
# ==========================================================
# We use Python to remove the model's "Chain of Thought" so the user only sees the final recommendation
final_output = re.sub(r'<reasoning>.*?</reasoning>', '[Internal Reasoning Hidden]', raw_answer, flags=re.DOTALL)

print("=" * 60)
print("Patient Symptoms: Fever, Cough, Oxygen Level = 88%")
print("=" * 60)
print("\nAssistant Output:\n")

# Print the filtered output without the reasoning!
print(final_output.strip())
print("\n" + "=" * 60) 


c:\Users\user\Downloads\AI-Training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:04<00:00, 68.38it/s]


Patient Symptoms: Fever, Cough, Oxygen Level = 88%

Assistant Output:

Step 1: Analyze the severity of each symptom (especially the oxygen level)
The patient has a fever, cough, and an oxygen level of 88%. The oxygen level is not normal, which could indicate respiratory distress or other complications that require immediate attention.

Step 2: Consider potential medical emergencies
Since the patient's oxygen level is not within normal range, it may be a sign of severe respiratory issues such as pneumonia, acute respiratory distress syndrome (ARDS), or even a life-threatening emergency like cardiac arrest. 

Step 3: Formulate a recommendation
Based on the patient's symptoms and potential medical emergencies, I recommend that the patient receives immediate medical attention to address their respiratory distress. This includes administering oxygen therapy, possibly in combination with other treatments such as antibiotics if necessary, and ensuring they receive proper care from a healthcar